In [ ]:
# inference/cli_demo.py

if __name__ == "__main__":
    parser = argparse.ArgumentParser(
        description="Generate a video from a text prompt using CogVideoX"
    )
    # =================================================================================================================================
    # Prompt Relay
    # modification-1
    # =================================================================================================================================
    parser.add_argument(
        "--prompt_filepath",
        type=str,
        default=None,
        help="The JSON file containing the global and local prompts with their temporal segments."
    )
    # =================================================================================================================================
    parser.add_argument(
        "--prompt", type=str, required=True, help="The description of the video to be generated"
    )

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

# modification-2
if args.prompt_filepath is not None:
        import json
        with open(args.prompt_filepath, 'r') as f:
            prompt_data = json.load(f)
        
        global_prompt = prompt_data.get("global_prompt", "")
        local_prompts = prompt_data.get("local_prompts", [])
        local_prompts = [" " + lp for lp in local_prompts]
        args.prompt = global_prompt + "".join(local_prompts)
        

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

# modification-3
generate_video(
    prompt=args.prompt,
    model_path=args.model_path,
    lora_path=args.lora_path,
    lora_rank=args.lora_rank,
    output_path=args.output_path,
    num_frames=args.num_frames,
    width=args.width,
    height=args.height,
    image_or_video_path=args.image_or_video_path,
    num_inference_steps=args.num_inference_steps,
    guidance_scale=args.guidance_scale,
    num_videos_per_prompt=args.num_videos_per_prompt,
    dtype=dtype,
    generate_type=args.generate_type,
    seed=args.seed,
    fps=args.fps,
    prompt_filepath = args.prompt_filepath # <--- Prompt Relay modification-3.1
)

def generate_video(
    prompt: str,
    model_path: str,
    lora_path: str = None,
    lora_rank: int = 128,
    num_frames: int = 81,
    width: Optional[int] = None,
    height: Optional[int] = None,
    output_path: str = "./output.mp4",
    image_or_video_path: str = "",
    num_inference_steps: int = 50,
    guidance_scale: float = 6.0,
    num_videos_per_prompt: int = 1,
    dtype: torch.dtype = torch.bfloat16,
    generate_type: str = Literal["t2v", "i2v", "v2v"],  # i2v: image to video, v2v: video to video
    seed: int = 42,
    fps: int = 16,
    prompt_filepath: Optional[str] = None # <--- Prompt Relay modification-3.2.0
):
       
# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

if generate_type == "i2v":
        video_generate = pipe(
            height=height,
            width=width,
            prompt=prompt,
            image=image,
            # The path of the image, the resolution of video will be the same as the image for CogVideoX1.5-5B-I2V, otherwise it will be 720 * 480
            num_videos_per_prompt=num_videos_per_prompt,  # Number of videos to generate per prompt
            num_inference_steps=num_inference_steps,  # Number of inference steps
            num_frames=num_frames,  # Number of frames to generate
            use_dynamic_cfg=True,  # This id used for DPM scheduler, for DDIM scheduler, it should be False
            guidance_scale=guidance_scale,
            generator=torch.Generator().manual_seed(seed),  # Set the seed for reproducibility
            prompt_filepath=prompt_filepath, # <--- Prompt Relay modification-3.2.1
        ).frames[0]
    elif generate_type == "t2v":
        video_generate = pipe(
            height=height,
            width=width,
            prompt=prompt,
            num_videos_per_prompt=num_videos_per_prompt,
            num_inference_steps=num_inference_steps,
            num_frames=num_frames,
            use_dynamic_cfg=True,
            guidance_scale=guidance_scale,
            generator=torch.Generator().manual_seed(seed),
            prompt_filepath=prompt_filepath, # <--- Prompt Relay modification-3.2.2
            fps=fps, # <--- Prompt Relay modification-15.2
        ).frames[0]
    else:
        video_generate = pipe(
            height=height,
            width=width,
            prompt=prompt,
            video=video,  # The path of the video to be used as the background of the video
            num_videos_per_prompt=num_videos_per_prompt,
            num_inference_steps=num_inference_steps,
            num_frames=num_frames,
            use_dynamic_cfg=True,
            guidance_scale=guidance_scale,
            generator=torch.Generator().manual_seed(seed),  # Set the seed for reproducibility
            prompt_filepath=prompt_filepath, # <--- Prompt Relay modification-3.2.3
        ).frames[0]

To edit the core CogVideoX files, you need to pull the Hugging Face diffusers repository and install it in "editable" mode. This ensures that any change you make to the code immediately applies to your Python environment without needing to reinstall.

```
git clone [https://github.com/huggingface/diffusers.git](https://github.com/huggingface/diffusers.git)
cd diffusers
pip install -e .
```

In [ ]:
# diffusers/src/diffusers/pipelines/cogvideo/pipeline_cogvideox.py
from typing import Optional # add

@torch.no_grad()
@replace_example_docstring(EXAMPLE_DOC_STRING)
def __call__(
    self,
    prompt: str | list[str] | None = None,
    negative_prompt: str | list[str] | None = None,
    height: int | None = None,
    width: int | None = None,
    num_frames: int | None = None,
    num_inference_steps: int = 50,
    timesteps: list[int] | None = None,
    guidance_scale: float = 6,
    use_dynamic_cfg: bool = False,
    num_videos_per_prompt: int = 1,
    eta: float = 0.0,
    generator: torch.Generator | list[torch.Generator] | None = None,
    latents: torch.FloatTensor | None = None,
    prompt_embeds: torch.FloatTensor | None = None,
    negative_prompt_embeds: torch.FloatTensor | None = None,
    output_type: str = "pil",
    return_dict: bool = True,
    attention_kwargs: dict[str, Any] | None = None,
    callback_on_step_end: Callable[[int, int], None] | PipelineCallback | MultiPipelineCallbacks | None = None,
    callback_on_step_end_tensor_inputs: list[str] = ["latents"],
    max_sequence_length: int = 226,
    prompt_filepath: Optional[str] = None, # <--- Prompt Relay modification-4
    fps: int = 16, # <--- Prompt Relay modification-15.1
)

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

# =================================================================================================================================
# Prompt Relay
# modification-5
# =================================================================================================================================
def _prepare_prompts(self, global_prompt, local_prompts, time_intervals, num_frames, fps, height, width):
        import math
        import torch
        
        # Hardware & Compression Constants
        vae_temporal_scale = getattr(self, "vae_scale_factor_temporal", 4)
        vae_spatial_scale = getattr(self, "vae_scale_factor_spatial", 8)
        patch_size = self.transformer.config.patch_size # usually 2

        latent_frames = (num_frames - 1) // vae_temporal_scale + 1
        h_lat = height // vae_spatial_scale
        w_lat = width // vae_spatial_scale
        h_patches = h_lat // patch_size
        w_patches = w_lat // patch_size
        tokens_per_frame = h_patches * w_patches

        full_prompt = global_prompt + "".join(local_prompts)
        
        # Tokenize and Find Sub-prompts
        full_ids = self.tokenizer(full_prompt, add_special_tokens=True, padding=False, return_attention_mask=False)["input_ids"]
        
        def sentence_to_token_indices(subsentences):
            def find_subsequence(haystack, needle):
                for start in range(len(haystack) - len(needle) + 1):
                    if haystack[start: start + len(needle)] == needle: 
                        return (start, start + len(needle))
                return None
                
            token_indices = {}
            for subsentence in subsentences:
                sub_ids = self.tokenizer(subsentence, padding=False, add_special_tokens=False)["input_ids"]
                match = find_subsequence(full_ids, sub_ids)
                if match is None:
                    raise ValueError(f"Subsentence not found in full prompt: {subsentence}")
                token_indices[subsentence] = match
            return token_indices

        # Build the payload containing the temporal cost data
        def build_q_token_idx(frame_intervals, token_spans, tokens_per_frame):        
            import math
            import torch
            
            q_token_idx = []
            epsilon = 0.1
            
            if len(frame_intervals) != 0:
                for _, (frame_start, frame_end, subsentences) in enumerate(frame_intervals): 
                    spans = []
                    for subsentence in subsentences:
                        start, end = token_spans[subsentence]
                        spans.extend(range(start, end))
                        
                    midpoint = (frame_start + frame_end) / 2.0
                    L = (frame_end - frame_start) / 2.0
                    
                    # Window parameter w = L - 2 
                    w = max(0.0, L - 2.0)
                    
                    # Ensures the attention prior reaches epsilon (0.1) at the endpoints
                    if L == w:
                        sigma_val = 0.1448 # Fallback for extremely short segments
                    else:
                        sigma_val = (L - w) / math.sqrt(2 * math.log(1 / epsilon))
                    
                    payload = {
                        "window": w,
                        "sigma": torch.tensor(sigma_val, dtype=torch.float16),
                        "midpoint": midpoint,
                        "tokens_per_frame": tokens_per_frame,
                        "local_token_idx": torch.tensor(spans, dtype=torch.long),
                    }
                    q_token_idx.append(payload)
            return q_token_idx

        spans = sentence_to_token_indices(local_prompts)
        
        # Interval Processing Logic
        frame_intervals = []
        if len(local_prompts) != 0:
            if time_intervals and len(time_intervals) > 0:
                # Map explicit time intervals (seconds) to latent frames
                for i, interval in enumerate(time_intervals):
                    start_sec, end_sec = interval
                    
                    # Convert seconds to raw frames based on fps
                    raw_start = math.floor(start_sec * fps)
                    raw_end = math.floor(end_sec * fps)
                    
                    # Cap raw_end at the maximum available raw frames
                    raw_end = min(raw_end, num_frames - 1)

                    patch_size_t = getattr(self.transformer.config, "patch_size_t", None)
                    temporal_patch_scale = patch_size_t if patch_size_t is not None else 1

                    # Map raw frames to 3D latent temporal patches
                    latent_start = (raw_start // vae_temporal_scale) // temporal_patch_scale
                    latent_end = (raw_end // vae_temporal_scale) // temporal_patch_scale
                    
                    # Prevent zero-length segments if intervals are extremely short
                    if latent_end == latent_start:
                        latent_end += 1
                        
                    frame_intervals.append((latent_start, latent_end, [local_prompts[i]]))
            else:
                num_prompts = len(local_prompts)
                frame_intervals = []
                for i in range(num_prompts):
                    l_start = round(i * latent_frames / num_prompts)
                    l_end = round((i + 1) * latent_frames / num_prompts)
                    frame_intervals.append((l_start, l_end, [local_prompts[i]]))
                
            q_token_idx = build_q_token_idx(frame_intervals, spans, tokens_per_frame)
        else:
            q_token_idx = None
            
        return q_token_idx
# =================================================================================================================================

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

# =================================================================================================================================
# Prompt Relay
# modification-6
# =================================================================================================================================
cross_attn_q_token_idx = None
if prompt_filepath is not None:
    import json
    with open(prompt_filepath, 'r') as f:
        prompts = json.load(f)

    global_prompt = prompts.get("global_prompt", "")
    local_prompts = prompts.get("local_prompts", [])
    # segment_lengths = prompts.get("segment_lengths", []) # use time_intervals now instead of specifying latent frames
    time_intervals = prompts.get("time_intervals", [])
    local_prompts = [" " + lp for lp in local_prompts]
    cross_attn_q_token_idx = self._prepare_prompts(
        global_prompt, local_prompts, time_intervals, num_frames, fps, height, width
    )
    
    prompt = global_prompt + "".join(local_prompts)
# =================================================================================================================================

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

with self.transformer.cache_context("cond_uncond"):
                    noise_pred = self.transformer(
                        hidden_states=latent_model_input,
                        encoder_hidden_states=prompt_embeds,
                        timestep=timestep,
                        image_rotary_emb=image_rotary_emb,
                        attention_kwargs=attention_kwargs,
                        cross_attn_q_token_idx=cross_attn_q_token_idx, # <--- Prompt Relay modification-7
                        return_dict=False,
                    )[0]

In [ ]:
# diffusers/src/diffusers/models/transformers/cogvideox_transformer_3d.py
from typing import Optional, List, Dict # add

@apply_lora_scale("attention_kwargs")
def forward(
    self,
    hidden_states: torch.Tensor,
    encoder_hidden_states: torch.Tensor,
    timestep: int | float | torch.LongTensor,
    timestep_cond: torch.Tensor | None = None,
    ofs: int | float | torch.LongTensor | None = None,
    image_rotary_emb: tuple[torch.Tensor, torch.Tensor] | None = None,
    attention_kwargs: dict[str, Any] | None = None,
    return_dict: bool = True,
    cross_attn_q_token_idx: Optional[List[Dict[str, Any]]] = None, # <--- Prompt Relay modification-8
)

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

for i, block in enumerate(self.transformer_blocks):
    if torch.is_grad_enabled() and self.gradient_checkpointing:
        hidden_states, encoder_hidden_states = self._gradient_checkpointing_func(
            block,
            hidden_states,
            encoder_hidden_states,
            emb,
            image_rotary_emb,
            attention_kwargs,
            cross_attn_q_token_idx, # <--- Prompt Relay modification-8.1
        )
    else:
        hidden_states, encoder_hidden_states = block(
            hidden_states=hidden_states,
            encoder_hidden_states=encoder_hidden_states,
            temb=emb,
            image_rotary_emb=image_rotary_emb,
            attention_kwargs=attention_kwargs,
            cross_attn_q_token_idx=cross_attn_q_token_idx, # <--- Prompt Relay modification-8.2
        )

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

def forward(
        self,
        hidden_states: torch.Tensor,
        encoder_hidden_states: torch.Tensor,
        temb: torch.Tensor,
        image_rotary_emb: tuple[torch.Tensor, torch.Tensor] | None = None,
        attention_kwargs: dict[str, Any] | None = None,
        cross_attn_q_token_idx: Optional[List[Dict[str, Any]]] = None, # <--- Prompt Relay modification-9
    ) -> tuple[torch.Tensor, torch.Tensor]:
        text_seq_length = encoder_hidden_states.size(1)
        attention_kwargs = attention_kwargs or {}

        # norm & modulate
        norm_hidden_states, norm_encoder_hidden_states, gate_msa, enc_gate_msa = self.norm1(
            hidden_states, encoder_hidden_states, temb
        )

        # attention
        attn_hidden_states, attn_encoder_hidden_states = self.attn1(
            hidden_states=norm_hidden_states,
            encoder_hidden_states=norm_encoder_hidden_states,
            image_rotary_emb=image_rotary_emb,
            **attention_kwargs,
            cross_attn_q_token_idx=cross_attn_q_token_idx, # <--- Prompt Relay modification-10
        )
# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

In [ ]:
# diffusers/src/diffusers/models/attention_processor.py

"""
Unlike Wan2.2, CogVideoX uses a Joint Attention block. It concatenates the text tokens and video tokens into a single sequence (text_seq_length + video_seq_length) before projecting them into queries, keys, and values.
Wan2.2 Prompt Relay build_temporal_cost and chunked_softmax_attention assume q contains only video frames and k contains only text.
If we apply torch.arange(Lq) naively, we will calculate frame temporal indices for text tokens, breaking the spatial-temporal mapping and corrupting Text $\rightarrow$ Text and Video $\rightarrow$ Video attention.
We must pass text_seq_length down into the chunked attention, generate a zero-mask of size [Lq, Lk] (which is effectively [Total_Seq, Total_Seq]), and apply the Gaussian decay only to the lower-left block where Video queries attend to Text keys.
"""

# =================================================================================================================================
# Prompt Relay
# modification-11
# =================================================================================================================================
def build_temporal_cost(q_token_idx, video_seq_length, text_seq_length, device, dtype):
    import torch
    
    # Allocate ONLY Video x Text penalty (~38 MB), bypassing the 14.7 GB Full map
    offset = torch.zeros(video_seq_length, text_seq_length, device=device, dtype=dtype)
    
    tokens_per_frame = int(q_token_idx[0]['tokens_per_frame'])
    
    query_frames = (
        torch.arange(video_seq_length, device=device, dtype=torch.long)
        // tokens_per_frame
    )

    for seg in q_token_idx:
        w = seg['window']
        sigma = seg['sigma'].clone().detach().to(device=device, dtype=torch.float32)
        local = seg['local_token_idx'].to(device=device)
        midpoint = torch.tensor(seg['midpoint'], dtype=torch.float32, device=device)

        d = (query_frames.float()[:, None] - midpoint).abs()
        cost = (torch.relu(d - w) ** 2) / (2 * sigma ** 2)

        # Apply cost directly (offset is now sized exclusively to video_seq_length)
        offset[:, local] = cost.to(offset.dtype)
        
    return offset
# =================================================================================================================================

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

# =================================================================================================================================
# Prompt Relay
# modification-12
# =================================================================================================================================
def chunked_softmax_attention(q, k, v, q_token_idx, text_seq_length, chunk_size=4096):
    import torch
    import torch.nn.functional as F

    B, H, Lq, D = q.shape
    video_seq_length = Lq - text_seq_length

    # Native SDPA for Text Queries
    q_text = q[:, :, :text_seq_length, :]
    out_text = F.scaled_dot_product_attention(
        q_text, k, v, dropout_p=0.0, is_causal=False
    )

    # Chunked SDPA for Video Queries
    q_video = q[:, :, text_seq_length:, :]
    out_video = torch.empty(B, H, video_seq_length, D, device=q.device, dtype=q.dtype)
    
    # Build the small Video -> Text penalty map
    temporal_cost_map = build_temporal_cost(q_token_idx, video_seq_length, text_seq_length, q.device, q.dtype)

    for start in range(0, video_seq_length, chunk_size):
        end = min(start + chunk_size, video_seq_length)
        q_chunk = q_video[:, :, start:end, :]
        chunk_len = end - start

        # The penalty applies ONLY to Text Keys. Video Keys have 0 penalty.
        penalty_text = temporal_cost_map[start:end] 
        # SDPA adds the mask to logits, so we must negate our penalty
        mask_text = -penalty_text.unsqueeze(0).unsqueeze(0) 

        # Apply CFG Leakage Fix (Zero out penalty for unconditional batch half)
        if B >= 2:
            zero_mask = torch.zeros_like(mask_text).repeat(B // 2, 1, 1, 1)
            cond_mask = mask_text.repeat(B - (B // 2), 1, 1, 1)
            mask_text = torch.cat([zero_mask, cond_mask], dim=0) 
        else:
            mask_text = mask_text.repeat(B, 1, 1, 1)

        # Video Keys get 0 mask
        mask_video = torch.zeros(B, 1, chunk_len, video_seq_length, device=q.device, dtype=q.dtype)

        # Stitch the mask together for the SDPA call [B, 1, chunk_len, L_total]
        mask_chunk = torch.cat([mask_text, mask_video], dim=-1)

        # Call ultra-fast native PyTorch C++ SDPA
        out_chunk = F.scaled_dot_product_attention(
            q_chunk, k, v, attn_mask=mask_chunk.to(q.dtype), dropout_p=0.0, is_causal=False
        )

        out_video[:, :, start:end, :] = out_chunk

        # Flush VRAM
        del mask_text, mask_video, mask_chunk, out_chunk

    return torch.cat([out_text, out_video], dim=2)
# =================================================================================================================================

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

def __call__(
        self,
        attn: Attention,
        hidden_states: torch.Tensor,
        encoder_hidden_states: torch.Tensor,
        attention_mask: torch.Tensor | None = None,
        image_rotary_emb: torch.Tensor | None = None,
        cross_attn_q_token_idx: Optional[List[Dict[str, Any]]] = None, # <--- Prompt Relay modification-13
    )

# ---------------------------------------------------------------------------------------------------------------------------------------------------------------

# =================================================================================================================================
# Prompt Relay
# modification-14
# =================================================================================================================================
if cross_attn_q_token_idx is not None and encoder_hidden_states is not None:
    # Bypass PyTorch SDPA and apply the temporal mask
    hidden_states = chunked_softmax_attention(query, key, value, cross_attn_q_token_idx, text_seq_length)
else:
    # Standard unmasked attention
    hidden_states = F.scaled_dot_product_attention(
        query, key, value, attn_mask=attention_mask, dropout_p=0.0, is_causal=False
    )
# =================================================================================================================================

What the Print Statement Proves

Successful Intercept: The control flow successfully bypassed PyTorch's native F.scaled_dot_product_attention and routed into your custom chunked_softmax_attention loop.

Shape Compatibility: The joint sequence length of 4276 tokens is passing through the chunking loop without tensor dimension mismatches or Out-Of-Memory (OOM) errors.

Active Mask Generation: Because temporal_cost_map.max() > 0 triggered, the script is actively calculating and applying mathematical penalties to the attention logits.

What the Print Statement Does NOT Prove (Potential Bugs)

Accurate Token Alignment: The T5 tokenizer frequently alters token boundaries by prepending meta-spaces. If your sentence_to_token_indices function misaligned the start and end indices by even a single position, the attention mask will penalize the wrong words entirely.

Effective Penalty Weighting: Your Gaussian decay parameters (sigma = 0.1448) were copied from Wan2.2's isolated cross-attention architecture. In CogVideoX's joint attention block, this specific penalty might be too weak to force visual changes, or too aggressive, destroying the video's spatial coherence.

Joint Attention Integrity: CogVideoX relies heavily on video tokens attending to other video tokens. If your offset masking logic accidentally spilled outside the specific text-to-video quadrant of the [4276, 4276] attention map, it will corrupt the underlying video structure, resulting in static or artifact-heavy frames

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    "THUDM/CogVideoX1.5-5B", 
    subfolder="tokenizer"
)

global_prompt = ""
local_prompts = [
    "A elderly man in a walks slowly up the grand, sweeping staircase of his opulent, empty mansion. Outside the adjacent tall glass window, a white seagull swoops ",
    "A elderly man in a walks slowly up the grand, sweeping staircase of his opulent, empty mansion. Outside the adjacent tall glass window, a white seagull swoops ",
    "A elderly man in a walks slowly up the grand, sweeping staircase of his opulent, empty mansion. Outside the adjacent tall glass window, a white seagull swoops ",
    "A elderly man in a walks slowly up the grand, sweeping staircase of his opulent, empty mansion. Outside the adjacent tall glass window, a white seagull swoops ",
    "A elderly man in a walks slowly up the grand, sweeping staircase of his opulent, empty mansion. Outside the adjacent tall glass window, a white seagull swoops ",
]
full_prompt = global_prompt + "".join(local_prompts)

tokens = tokenizer(full_prompt, add_special_tokens=True, padding=False, return_attention_mask=False)
token_count = len(tokens["input_ids"])

print(f"Total tokens: {token_count} / 224")

Total tokens: 102 / 224
